In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll2ndIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("uptake_results.xlsx")   # <-- Excel file with all tables
figName1     = Path("SecretionProfile2ndIter.pdf")
figName2     = Path("SecretionMatrix2ndIter.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "products"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll1stIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("SecretionProfile1stIterUnionMatrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("SecretionProfile1stIterUnion.pdf")
figName2     = Path("SecretionMatrix1stIterUnion.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "products"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll100_1stIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("SecretionProfile1stIter100Matrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("SecretionProfile1stIter100.pdf")
figName2     = Path("SecretionMatrix1stIter100.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "products"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll100_2ndIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("SecretionProfile2ndIter100Matrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("SecretionProfile2ndIter100.pdf")
figName2     = Path("SecretionMatrix2ndIter100.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "products"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


Uptakes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll2ndIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("UptakeProfile2ndIterUnionMatrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("UptakeProfile2ndIterUnion.pdf")
figName2     = Path("UptakeMatrix2ndIterUnion.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "substrates"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll1stIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("UptakeProfile1stIterUnionMatrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("UptakeProfile1stIterUnion.pdf")
figName2     = Path("UptakeMatrix1stIterUnion.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "substrates"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll100_1stIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("UptakeProfile1stIter100Matrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("UptakeProfile1stIter100.pdf")
figName2     = Path("UptakeMatrix1stIter100.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "substrates"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict
import matplotlib

# -------------------------
# Paths (edit if needed)
# -------------------------
summary_path = Path("summaryAll100_2ndIteration.xlsx")
class_path   = Path("CompoundClassification.xlsx")
out_xlsx     = Path("UptakeProfile2ndIter100Matrix.xlsx")   # <-- Excel file with all tables
figName1     = Path("UptakeProfile2ndIter100.pdf")
figName2     = Path("UptakeMatrix2ndIter100.pdf")

# -------------------------
# Parameters
# -------------------------
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "substrates"       # KEGG IDs column in the summary sheets

# Use the 'Graphing' sheet which has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# Optionally limit compounds in the fertility heatmap for readability (None = show all)
TOP_N_COMPOUNDS = None  # e.g., 200

# -------------------------
# Helpers
# -------------------------
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# -------------------------
# 1) Load mapping from 'Graphing' sheet (preserve exact Class/Name strings)
# -------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

# Clean IDs only (for reliable joins); DO NOT alter Class/Name text
graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

# Lookups — use Excel text verbatim
id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))

# -------------------------
# 2) Read species sheets → presence matrix
# -------------------------
xl = pd.ExcelFile(summary_path)
species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

species_to_ids = {}
missing_col_sheets = []
for sheet in species_sheets:
    df = pd.read_excel(summary_path, sheet_name=sheet)
    col = pick_id_col(df, summary_id_col_hint)
    if col is None:
        missing_col_sheets.append(sheet)
        continue
    sp = sheet[:-len(summary_sheet_suffix)]
    species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

if missing_col_sheets:
    print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
# drop background and keep only IDs we can classify
all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
for sp, ids in species_to_ids.items():
    on = [i for i in ids if i in presence.columns]
    presence.loc[sp, on] = 1

# -------------------------
# 3) Fertilities
# -------------------------
n_species = presence.shape[0]
compound_fertility = presence.sum(axis=0) / float(n_species)        # fraction of species per compound
sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}  # compound → class (verbatim labels)

# Class fertility = mean compound fertility within each class
class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)
class_order = class_fertility.index.tolist()

# Precompute columns for each class
class_to_cols = defaultdict(list)
for sid, cls in sub_to_class.items():
    class_to_cols[cls].append(sid)

# -------------------------
# 4) Per-class uptake matrix (same as original)
# -------------------------
class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
for sp in class_matrix.index:
    row = presence.loc[sp]
    for cls in class_order:
        cols = class_to_cols.get(cls, [])
        class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

# Breadth scores & species order (versatility) — LOW → HIGH
class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
species_order = class_matrix.index

# -------------------------
# 5) CLASS-LEVEL HEATMAP (classes ordered by class fertility) — NO PERCENTAGES
# -------------------------
plot_df = class_matrix.loc[species_order, class_order]
data = plot_df.T.values
x = np.arange(data.shape[1] + 1)
y = np.arange(data.shape[0] + 1)

plt.figure(figsize=(12, 8))
plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
# y-axis: class names exactly as in Excel (no %)
plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
plt.title("Class Uptake Heatmap (Classes ordered by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Product Classes (high fertility → low)")
cb = plt.colorbar(); cb.set_label("Fraction Uptake (0–1)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName1, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 6) COMPOUND-LEVEL HEATMAP (use compound NAMES + fertility)
# -------------------------
cf_sorted = compound_fertility.sort_values(ascending=False)
if TOP_N_COMPOUNDS is not None:
    cf_sorted = cf_sorted.iloc[:TOP_N_COMPOUNDS]

heat_df = presence.loc[species_order, cf_sorted.index].T
data2 = heat_df.values
x2 = np.arange(data2.shape[1] + 1)
y2 = np.arange(data2.shape[0] + 1)

plt.figure(figsize=(max(12, data2.shape[1]*0.35), max(8, data2.shape[0]*0.02 + 6)))
plt.pcolormesh(x2, y2, data2, cmap="Greys", shading="auto")
plt.xticks(np.arange(data2.shape[1]) + 0.5, heat_df.columns, rotation=90, fontsize=6)

# y-axis labels show compound names exactly as in Excel (+ % fertility for readability)
y2_labels = [f"{id_to_name.get(cid, cid)} ({cf_sorted[cid]*100:.0f}%)" for cid in heat_df.index]
plt.yticks(np.arange(data2.shape[0]) + 0.5, y2_labels, fontsize=6)

plt.title("Compound Presence Heatmap (by fertility)", fontsize=12)
plt.xlabel("Species (low versatility → high)")
plt.ylabel("Compounds (high fertility → low)")
cb2 = plt.colorbar(); cb2.set_label("Presence (1) / Absence (0)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(figName2, dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 7) SAVE ALL CALCULATIONS / TABLES TO ONE EXCEL FILE
# -------------------------
compound_fertility_df = pd.DataFrame({
    "KEGG ID": cf_sorted.index,
    "Name": [id_to_name.get(i, i) for i in cf_sorted.index],        # names verbatim from Excel
    "Class": [sub_to_class.get(i, None) for i in cf_sorted.index],  # classes verbatim from Excel
    "Fertility": cf_sorted.values
})

class_fertility_df = class_fertility.rename("Fertility").reset_index().rename(columns={"index": "Class"})

species_breadth_df = class_matrix[["Breadth Score", "Normalized Breadth Score"]].copy()

# Presence matrix ordered for readability: species low→high, compounds high→low fertility
presence_out = presence.loc[species_order, cf_sorted.index]

# Mapping actually used (filtered to presence columns)
mapping_used_df = pd.DataFrame({
    "KEGG ID": presence_out.columns,
    "Name": [id_to_name.get(i, i) for i in presence_out.columns],
    "Class": [sub_to_class.get(i, None) for i in presence_out.columns],
})

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    class_matrix.to_excel(writer, sheet_name="ClassMatrix")
    species_breadth_df.to_excel(writer, sheet_name="SpeciesBreadth")
    compound_fertility_df.to_excel(writer, sheet_name="CompoundFertility", index=False)
    class_fertility_df.to_excel(writer, sheet_name="ClassFertility", index=False)
    presence_out.to_excel(writer, sheet_name="PresenceMatrix")
    mapping_used_df.to_excel(writer, sheet_name="MappingUsed", index=False)

print(f"Saved calculations/tables to: {out_xlsx.resolve()}")



In [ ]:
#!/usr/bin/env python3
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
# ============================================================
# 0. Input files
# ============================================================

FILE_SUMMARY_1 = "summaryAll1stIteration.xlsx"   # Basal
FILE_SUMMARY_2 = "summaryAll2ndIteration.xlsx"   # Conditioned
FILE_CLASSES   = "CompoundClassification.xlsx"   # KEGG → Classification

# ============================================================
# 1. Load summary files and collect counts per model
# ============================================================

# Basal and conditioned media summary files
xls_sa1 = pd.ExcelFile(FILE_SUMMARY_1)   # Basal
xls_sa2 = pd.ExcelFile(FILE_SUMMARY_2)   # Conditioned


def species_name_from_sheet(sheet_name):
    """
    Convert sheet name to species/model ID.
    Example: 'bbet_summary' -> 'bbet'
    """
    if sheet_name.endswith("_summary"):
        return sheet_name[:-8]
    return sheet_name


def collect_counts(xls):
    """
    For a summaryAll* file (multiple sheets), return:
      counts_products  : dict {species -> # unique secreted compounds}
      counts_substrates: dict {species -> # unique uptaken compounds}
    Assumes each sheet has columns 'products' and 'substrates'.
    """
    counts_products = {}
    counts_substrates = {}

    for sheet in xls.sheet_names:
        sp = species_name_from_sheet(sheet)
        df = pd.read_excel(xls, sheet_name=sheet)

        # unique secreted compounds (products)
        if "products" in df.columns:
            prods = df["products"].dropna().unique()
            counts_products[sp] = len(prods)

        # unique uptaken compounds (substrates)
        if "substrates" in df.columns:
            subs = df["substrates"].dropna().unique()
            counts_substrates[sp] = len(subs)

    return counts_products, counts_substrates


# Basal (1st iteration) — organism-level
cp1, cu1 = collect_counts(xls_sa1)   # cp1 = products, cu1 = uptakes

# Conditioned (2nd iteration) — organism-level
cp2, cu2 = collect_counts(xls_sa2)   # cp2 = products, cu2 = uptakes


# ============================================================
# 1b. For points 3 & 4: build compound -> set(species) maps
# ============================================================

def collect_compound_species_maps(xls):
    """
    For a summaryAll* file, build:
      - substrates_to_species: {substrate -> set of species that uptake it}
      - products_to_species  : {product   -> set of species that secrete it}
    """
    substrates_to_species = {}
    products_to_species = {}

    for sheet in xls.sheet_names:
        sp = species_name_from_sheet(sheet)
        df = pd.read_excel(xls, sheet_name=sheet)

        # Substrates: compounds taken up by this species
        if "substrates" in df.columns:
            for sub in df["substrates"].dropna().unique():
                substrates_to_species.setdefault(sub, set()).add(sp)

        # Products: compounds secreted by this species
        if "products" in df.columns:
            for prod in df["products"].dropna().unique():
                products_to_species.setdefault(prod, set()).add(sp)

    return substrates_to_species, products_to_species


# Basal — compound-level
subs_to_species_1, prods_to_species_1 = collect_compound_species_maps(xls_sa1)

# Conditioned — compound-level
subs_to_species_2, prods_to_species_2 = collect_compound_species_maps(xls_sa2)


def species_counts_from_map(compound_to_species):
    """
    Convert {compound -> set(species)} into {compound -> #species}.
    """
    return {c: len(sp_set) for c, sp_set in compound_to_species.items()}


# Point 3: Substrate fertility (per metabolite, per condition)
#   "Number of species that uptake a given substrate"
subs_fertility_1 = species_counts_from_map(subs_to_species_1)  # basal
subs_fertility_2 = species_counts_from_map(subs_to_species_2)  # conditioned

# Point 4: Substrate versatility (per metabolite, per condition)
#   "Number of species producing a given metabolite"
prods_versatility_1 = species_counts_from_map(prods_to_species_1)  # basal
prods_versatility_2 = species_counts_from_map(prods_to_species_2)  # conditioned


# ============================================================
# 2. Build totals + % change for ordering
# ============================================================

def build_totals(counts1, counts2):
    """
    Generic utility for species, substrates, products, or classes.

    counts1: dict {ID -> count} for 1st iteration (basal)
    counts2: dict {ID -> count} for 2nd iteration (conditioned)

    ID can be a species, a substrate (KEGG ID), a product, or a class.

    Returns a DataFrame indexed by ID with columns:
      - total_1   : basal count
      - total_2   : conditioned count
      - pct_change: (total_2 - total_1)/total_1 * 100
    Sorted by pct_change in descending order.
    """
    ids_all = sorted(set(counts1) | set(counts2))
    rows = []

    for _id in ids_all:
        t1 = counts1.get(_id, 0)
        t2 = counts2.get(_id, 0)

        if t1 == 0 and t2 == 0:
            pct = 0.0
        elif t1 == 0 and t2 > 0:
            pct = math.inf  # infinite relative increase
        else:
            pct = (t2 - t1) / float(t1) * 100.0

        rows.append((_id, t1, t2, pct))

    df = pd.DataFrame(rows, columns=["ID", "total_1", "total_2", "pct_change"])
    df = df.set_index("ID")
    df = df.sort_values("pct_change", ascending=False)

    # Sanity check: monotone in pct_change
    vals = df["pct_change"].values
    assert all(vals[i] >= vals[i+1] or math.isinf(vals[i])
               for i in range(len(vals)-1)), "Ordering by pct_change failed!"

    print("\n=== entries, sorted by % change ===")
    print(df)

    return df


# ---------- Species-centric (points 1 & 2) ----------
tot_uptake    = build_totals(cu1, cu2)   # Organism versatility
tot_secretion = build_totals(cp1, cp2)   # Organism fertility (amount)

# ---------- Compound-centric (points 3 & 4, metabolite level) ----------
tot_substrate_fertility   = build_totals(subs_fertility_1, subs_fertility_2)
tot_substrate_versatility = build_totals(prods_versatility_1, prods_versatility_2)


# ============================================================
# 3. Plot stacked bars and save as PDF
# ============================================================

def format_pct(p):
    """Nicely format percentage change for labels."""
    if math.isinf(p):
        return "∞%"
    return f"{p:+.0f}%"


def plot_stacked_and_save(label,
                          totals,
                          pdf_filename,
                          x_label="Count",
                          max_items=None):
    """
    Create a stacked horizontal barplot (Basal + Conditioned),
    ordered by % change, and save as PDF.

    Works for:
      - species-based metrics (1 & 2)
      - compound-based metrics (3 & 4)
      - class-based metrics (aggregated later)
    """
    # Ensure correct ordering
    totals = totals.sort_values("pct_change", ascending=False)

    # Optionally restrict to top N entries (e.g. for metabolites)
    if max_items is not None and len(totals) > max_items:
        totals = totals.iloc[:max_items].copy()

    # Labels with % change
    entry_labels = [
        f"{idx} ({format_pct(p)})"
        for idx, p in zip(totals.index, totals["pct_change"])
    ]

    y = np.arange(len(totals))
    basal = totals["total_1"].values
    cond  = totals["total_2"].values

    fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(totals))))

    # Basal segment
    ax.barh(y, basal, color="lightgrey", label="Basal")

    # Conditioned segment stacked on top
    ax.barh(y, cond, left=basal, color="black", label="Conditioned")

    ax.set_xlabel(x_label)
    ax.set_ylabel("Entries (ordered by % change)")
    ax.set_yticks(y)
    ax.set_yticklabels(entry_labels)
    ax.invert_yaxis()  # Top = largest relative increase

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, loc="lower right")

    fig.suptitle(
        f"{label}\n"
        "Stacked Basal + Conditioned, ordered by % change (Basal → Conditioned)",
        y=1.02
    )

    plt.tight_layout()
    fig.savefig(pdf_filename, bbox_inches="tight")
    print("Saved {} figure to: {}".format(label, pdf_filename))
    plt.show()

    return fig


# --------- FINAL PLOTS: SPECIES-LEVEL (points 1 & 2) ---------

fig_uptake_stacked = plot_stacked_and_save(
    "Uptakes per model",
    tot_uptake,
    "Uptakes_stacked_Basal_vs_Conditioned.pdf",
    x_label="Number of unique substrates"
)

fig_secretion_stacked = plot_stacked_and_save(
    "Secretions per model",
    tot_secretion,
    "Secretions_stacked_Basal_vs_Conditioned.pdf",
    x_label="Number of unique products"
)


# --------- FINAL PLOTS: METABOLITE-LEVEL (points 3 & 4) ---------

fig_substrate_fertility = plot_stacked_and_save(
    "Substrate fertility\n(# species uptaking each substrate)",
    tot_substrate_fertility,
    "SubstrateFertility_species_per_substrate.pdf",
    x_label="Number of species",
    max_items=40  # top metabolites by % change; set to None to show all
)

fig_substrate_versatility = plot_stacked_and_save(
    "Substrate versatility\n(# species producing each product)",
    tot_substrate_versatility,
    "SubstrateVersatility_species_per_product.pdf",
    x_label="Number of species",
    max_items=40  # top metabolites by % change; set to None to show all
)


# ============================================================
# 4. Class-level substrate fertility & versatility
#    (x-axis = number of species, not species×metabolites)
# ============================================================

# Load KEGG → Classification table
class_df = pd.read_excel(FILE_CLASSES)
kegg_to_class = dict(zip(class_df["KEGG ID"], class_df["Classification"]))


def build_class_to_species(compound_to_species, mapping, default_class="Unclassified"):
    """
    From {metabolite -> set(species)} build:
        {class -> set(species that use at least one metabolite in this class)}
    """
    class_to_species = {}

    for met, sp_set in compound_to_species.items():
        cls = mapping.get(met, default_class)
        if cls not in class_to_species:
            class_to_species[cls] = set()
        # union: a species is counted once per class,
        # even if it uses many metabolites in that class
        class_to_species[cls].update(sp_set)

    return class_to_species


def class_species_totals(compound_to_species_1,
                         compound_to_species_2,
                         mapping):
    """
    Build a class-level DataFrame with:
        total_1   = # species using at least one metabolite of the class (Basal)
        total_2   = same for Conditioned
        pct_change = relative change
    """
    class_to_species_1 = build_class_to_species(compound_to_species_1, mapping)
    class_to_species_2 = build_class_to_species(compound_to_species_2, mapping)

    counts1 = {cls: len(sps) for cls, sps in class_to_species_1.items()}
    counts2 = {cls: len(sps) for cls, sps in class_to_species_2.items()}

    # reuse generic build_totals (IDs are class names)
    return build_totals(counts1, counts2)


# ---------- Class-level 3: Substrate fertility per class ----------
#    "# species uptaking at least one substrate from the class"
class_substrate_fertility = class_species_totals(
    subs_to_species_1,
    subs_to_species_2,
    kegg_to_class
)

fig_sf_class = plot_stacked_and_save(
    "Substrate fertility per class\n(# species uptaking ≥1 substrate of the class)",
    class_substrate_fertility,
    "SubstrateFertility_perClass_species.pdf",
    x_label="Number of species",
    max_items=None
)

# ---------- Class-level 4: Substrate versatility per class ----------
#    "# species producing at least one product from the class"
class_substrate_versatility = class_species_totals(
    prods_to_species_1,
    prods_to_species_2,
    kegg_to_class
)

fig_sv_class = plot_stacked_and_save(
    "Substrate versatility per class\n(# species producing ≥1 product of the class)",
    class_substrate_versatility,
    "SubstrateVersatility_perClass_species.pdf",
    x_label="Number of species",
    max_items=None
)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict

# ==========================
# Paths
# ==========================
summary1_path = Path("summaryAll1stIteration.xlsx")   # Basal
summary2_path = Path("summaryAll2ndIteration.xlsx")   # Conditioned
class_path    = Path("CompoundClassification.xlsx")

fig_basal     = Path("UptakeProfile1stIterUnion.pdf")
fig_cond      = Path("UptakeProfile2ndIterUnion.pdf")

# ==========================
# Parameters
# ==========================
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "substrates"     # KEGG IDs column in the summary sheets

# 'Graphing' sheet has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# ==========================
# Helpers
# ==========================
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\\d{5}$").mean() > 0.5):
            return c
    return None

# --------------------------
# 1) Load mapping from 'Graphing' sheet
# --------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))


# ==========================
# Core function: one iteration → one orange heatmap
# ==========================
def build_class_heatmap(summary_path: Path,
                        fig_path: Path,
                        title: str,
                        class_order_ref=None):
    """
    Build presence + class_matrix for a given summaryAll*.xlsx
    and plot the orange class-level heatmap.

    - class_order_ref: if given, use this EXACT class order.
                       missing classes get all zeros (white rows).
    - species order is ALWAYS recomputed from this iteration's
      Normalized Breadth Score (low → high).
    """
    # ---- Read species sheets -> presence matrix ----
    xl = pd.ExcelFile(summary_path)
    species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

    species_to_ids = {}
    missing_col_sheets = []

    for sheet in species_sheets:
        df = pd.read_excel(summary_path, sheet_name=sheet)
        col = pick_id_col(df, summary_id_col_hint)
        if col is None:
            missing_col_sheets.append(sheet)
            continue
        sp = sheet[:-len(summary_sheet_suffix)]
        species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

    if missing_col_sheets:
        print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

    all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
    # drop background and keep only IDs we can classify
    all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

    presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
    for sp, ids in species_to_ids.items():
        on = [i for i in ids if i in presence.columns]
        presence.loc[sp, on] = 1

    # ---- Fertilities & class mapping ----
    n_species = presence.shape[0]
    compound_fertility = presence.sum(axis=0) / float(n_species)
    sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}

    # class fertility for THIS iteration
    class_fertility = compound_fertility.groupby(pd.Series(sub_to_class)).mean().sort_values(ascending=False)

    # Class order:
    #  - if ref is None → this iteration defines it
    #  - if ref is given → force this order (for basal), even if some classes are all-zero
    if class_order_ref is None:
        class_order = class_fertility.index.tolist()
    else:
        class_order = list(class_order_ref)

    # Precompute columns for each class
    from collections import defaultdict
    class_to_cols = defaultdict(list)
    for sid, cls in sub_to_class.items():
        class_to_cols[cls].append(sid)

    # ---- Per-class uptake matrix ----
    class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
    for sp in class_matrix.index:
        row = presence.loc[sp]
        for cls in class_order:
            cols = class_to_cols.get(cls, [])
            class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

    # ---- Species order: ALWAYS from this iteration's breadth ----
    class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
    class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
    class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
    species_order = class_matrix.index

    # ---- CLASS-LEVEL HEATMAP ----
    plot_df = class_matrix.loc[species_order, class_order]
    data = plot_df.T.values
    x = np.arange(data.shape[1] + 1)
    y = np.arange(data.shape[0] + 1)

    plt.figure(figsize=(12, 8))
    plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
    plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
    plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
    plt.title(title, fontsize=12)
    plt.xlabel("Species (low versatility → high, this iteration only)")
    plt.ylabel("Product Classes (class order from 2nd iter)")
    cb = plt.colorbar()
    cb.set_label("Fraction Uptake (0–1)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved heatmap to: {fig_path.resolve()}")

    # return class order so we can reuse it for basal
    return class_order


# ==========================
# Run: 2nd iteration defines class order; 1st uses same class order
# ==========================

# 2nd iteration (conditioned) — defines the reference CLASS order
class_order_2 = build_class_heatmap(
    summary2_path,
    fig_cond,
    title="Class Uptake Heatmap – 2nd Iteration (Conditioned)",
    class_order_ref=None
)

# 1st iteration (basal) — uses SAME class order, but its OWN species order
_ = build_class_heatmap(
    summary1_path,
    fig_basal,
    title="Class Uptake Heatmap – 1st Iteration (Basal)\n(Class rows aligned to 2nd Iteration)",
    class_order_ref=class_order_2
)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional
from collections import defaultdict

# ==========================
# Paths
# ==========================
summary1_path = Path("summaryAll1stIteration.xlsx")   # Basal
summary2_path = Path("summaryAll2ndIteration.xlsx")   # Conditioned
class_path    = Path("CompoundClassification.xlsx")

fig_basal_products = Path("ProductProfile1stIterUnion.pdf")
fig_cond_products  = Path("ProductProfile2ndIterUnion.pdf")

# ==========================
# Parameters
# ==========================
summary_sheet_suffix = "_summary"       # species sheets end with this
summary_id_col_hint  = "products"       # *** NOW USING PRODUCTS *** (secreted KEGG IDs)

# 'Graphing' sheet has KEGG ID, Name, Classification
class_sheet    = "Graphing"
class_id_col   = "KEGG ID"
class_name_col = "Name"
class_col      = "Classification"

# IDs to drop (background)
drop_ids = {"C00001"}

# ==========================
# Helpers
# ==========================
def pick_id_col(df: pd.DataFrame, hint: str) -> Optional[str]:
    """Pick the KEGG-ID column from a species sheet."""
    if hint in df.columns:
        return hint
    # case-insensitive match
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs (Cxxxxx)
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\\d{5}$").mean() > 0.5):
            return c
    return None

# --------------------------
# 1) Load mapping from 'Graphing' sheet
# --------------------------
graph_df = pd.read_excel(
    class_path,
    sheet_name=class_sheet,
    usecols=[class_id_col, class_name_col, class_col]
).dropna(subset=[class_id_col, class_name_col, class_col])

graph_df[class_id_col] = graph_df[class_id_col].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=[class_id_col], keep="first")

id_to_class = dict(zip(graph_df[class_id_col], graph_df[class_col]))
id_to_name  = dict(zip(graph_df[class_id_col], graph_df[class_name_col]))


# ==========================
# Core function: one iteration → one orange heatmap (PRODUCTS)
# ==========================
def build_class_heatmap_products(summary_path: Path,
                                 fig_path: Path,
                                 title: str,
                                 class_order_ref=None):
    """
    Build presence + class_matrix for a given summaryAll*.xlsx
    using PRODUCTS (secreted compounds), and plot the orange
    class-level heatmap.

    - class_order_ref: if given, use this EXACT class order.
                       missing classes get all zeros (white rows).
    - species order is ALWAYS recomputed from this iteration's
      Normalized Breadth Score (low → high).
    """
    # ---- Read species sheets -> presence matrix ----
    xl = pd.ExcelFile(summary_path)
    species_sheets = [s for s in xl.sheet_names if s.endswith(summary_sheet_suffix)]

    species_to_ids = {}
    missing_col_sheets = []

    for sheet in species_sheets:
        df = pd.read_excel(summary_path, sheet_name=sheet)
        col = pick_id_col(df, summary_id_col_hint)   # now hint="products"
        if col is None:
            missing_col_sheets.append(sheet)
            continue
        sp = sheet[:-len(summary_sheet_suffix)]
        species_to_ids[sp] = df[col].dropna().astype(str).str.strip().tolist()

    if missing_col_sheets:
        print("Warning: could not detect a KEGG ID column in these sheets:", missing_col_sheets)

    all_ids = sorted({i for ids in species_to_ids.values() for i in ids})
    # drop background and keep only IDs we can classify
    all_ids = [i for i in all_ids if i not in drop_ids and i in id_to_class]

    presence = pd.DataFrame(0, index=sorted(species_to_ids), columns=all_ids, dtype=int)
    for sp, ids in species_to_ids.items():
        on = [i for i in ids if i in presence.columns]
        presence.loc[sp, on] = 1

    # ---- Fertilities & class mapping ----
    n_species = presence.shape[0]
    compound_fertility = presence.sum(axis=0) / float(n_species)
    sub_to_class = {sid: id_to_class[sid] for sid in presence.columns}

    # class fertility for THIS iteration
    class_fertility = (
        compound_fertility.groupby(pd.Series(sub_to_class))
        .mean()
        .sort_values(ascending=False)
    )

    # Class order:
    #  - if ref is None → this iteration defines it
    #  - if ref is given → force this order (for basal), even if some classes are all-zero
    if class_order_ref is None:
        class_order = class_fertility.index.tolist()
    else:
        class_order = list(class_order_ref)

    # Precompute columns for each class
    class_to_cols = defaultdict(list)
    for sid, cls in sub_to_class.items():
        class_to_cols[cls].append(sid)

    # ---- Per-class secretion matrix ----
    class_matrix = pd.DataFrame(0.0, index=presence.index, columns=class_order)
    for sp in class_matrix.index:
        row = presence.loc[sp]
        for cls in class_order:
            cols = class_to_cols.get(cls, [])
            class_matrix.at[sp, cls] = (row[cols].sum() / len(cols)) if cols else 0.0

    # ---- Species order: ALWAYS from this iteration's breadth ----
    class_matrix["Breadth Score"] = class_matrix[class_order].sum(axis=1)
    class_matrix["Normalized Breadth Score"] = class_matrix["Breadth Score"] / len(class_order)
    class_matrix.sort_values("Normalized Breadth Score", ascending=True, inplace=True)
    species_order = class_matrix.index

    # ---- CLASS-LEVEL HEATMAP ----
    plot_df = class_matrix.loc[species_order, class_order]
    data = plot_df.T.values
    x = np.arange(data.shape[1] + 1)
    y = np.arange(data.shape[0] + 1)

    plt.figure(figsize=(12, 8))
    plt.pcolormesh(x, y, data, cmap="Oranges", shading="auto")
    plt.xticks(np.arange(data.shape[1]) + 0.5, plot_df.index, rotation=90, fontsize=6)
    plt.yticks(np.arange(data.shape[0]) + 0.5, class_order, fontsize=8)
    plt.title(title, fontsize=12)
    plt.xlabel("Species (low versatility → high, this iteration)")
    plt.ylabel("Product Classes (class order from 2nd iter)")
    cb = plt.colorbar()
    cb.set_label("Fraction Secretion (0–1)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved product-class heatmap to: {fig_path.resolve()}")

    return class_order


# ==========================
# Run: 2nd iteration defines class order; 1st uses same class order
# ==========================

# 2nd iteration (conditioned) — defines the reference CLASS order for PRODUCTS
class_order_products_2 = build_class_heatmap_products(
    summary2_path,
    fig_cond_products,
    title="Class Secretion Heatmap – 2nd Iteration (Conditioned, Products)",
    class_order_ref=None
)

# 1st iteration (basal) — uses SAME product-class order, but its OWN species order
_ = build_class_heatmap_products(
    summary1_path,
    fig_basal_products,
    title="Class Secretion Heatmap – 1st Iteration (Basal, Products)\n(Class rows aligned to 2nd Iteration)",
    class_order_ref=class_order_products_2
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, PathPatch
from matplotlib.path import Path
import re
import os
import matplotlib
# ============================================================
# CONFIG
# ============================================================

SUMMARY1_PATH = "summaryAll1stIteration.xlsx"   # 1st iteration
SUMMARY2_PATH = "summaryAll2ndIteration.xlsx"   # 2nd iteration
COMPOUND_CLASS_PATH = "CompoundClassification.xlsx"
MIN_PRODUCERS = 1
MIN_CONSUMERS = 1
MAX_METABOLITES_PER_CLASS = 10

EDGE_BASE_WIDTH = 1.0
EDGE_SCALE = 3.0
EDGE_COLOR = (0/255, 170/255, 170/255, 0.45)  # teal RGBA

OUTPUT_DIR = "sankey_pdfs_iter1_to_iter2_newMets_noSelf"

phyla_dict = {
    "cohm": "Firmicutes",
    "cuvk": "Actinobacteria",
    "cxy": "Actinobacteria",
    "mip": "Actinobacteria",
    "rfa": "Actinobacteria",
    "cpi": "Bacteroidetes",
    "flk": "Bacteroidetes",
    "mgk": "Bacteroidetes",
    "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria",
    "cauj": "Alphaproteobacteria",
    "dri": "Alphaproteobacteria",
    "lgu": "Gammaproteobacteria",
    "lur": "Gammaproteobacteria",
    "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria",
    "pscf": "Gammaproteobacteria",
    "pzu": "Alphaproteobacteria",
    "racd": "Gammaproteobacteria",
    "tro": "Alphaproteobacteria",
    "vpe": "Betaproteobacteria",
}

phyla_order = [
    "Alphaproteobacteria",
    "Gammaproteobacteria",
    "Actinobacteria",
    "Bacteroidetes",
    "Betaproteobacteria",
    "Firmicutes",
]

phyla_color_map = {
    "Alphaproteobacteria": "#E41A1C",
    "Gammaproteobacteria": "#377EB8",
    "Actinobacteria":      "#4DAF4A",
    "Bacteroidetes":       "#984EA3",
    "Betaproteobacteria":  "#FF7F00",
    "Firmicutes":          "#A65628",
    "Unknown":             "#999999",
}

plt.style.use("default")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# A. Build edges using iter1 + iter2
#    AND: new metabolites only
#    AND: drop species that both produce & consume same met
# ============================================================

def build_edges(summary1_path,
                summary2_path,
                compound_class_path,
                min_producers=1,
                min_consumers=1):
    """
    Producers: species producing m in iter1 (products).
    Consumers: species consuming m in iter2 (substrates).

    Metabolites must:
      - be produced in iter1,
      - be consumed in iter2,
      - NOT be substrates in iter1 (new),
      - AND after removing species that both produce & consume m,
        still have >= min_producers and >= min_consumers.

    Species that are both producer and consumer for m are removed
    from BOTH sides for that m (no connection to that metabolite at all).
    """

    # ---- iter1: products (producers) + substrates (for exclusion) ----
    xls1 = pd.ExcelFile(summary1_path)
    sheets1 = xls1.sheet_names

    producer_map = {}      # KEGG -> set(species)
    substrates_iter1 = set()

    for sheet in sheets1:
        df = pd.read_excel(xls1, sheet)
        species = sheet.replace("_summary", "")

        if "products" not in df.columns:
            raise ValueError(f"Sheet {sheet} in iter1 missing 'products'")
        for prod in df["products"].dropna().unique():
            producer_map.setdefault(prod, set()).add(species)

        if "substrates" in df.columns:
            for sub in df["substrates"].dropna().unique():
                substrates_iter1.add(sub)

    # ---- iter2: substrates (consumers) ----
    xls2 = pd.ExcelFile(summary2_path)
    sheets2 = xls2.sheet_names

    consumer_map = {}   # KEGG -> set(species)

    for sheet in sheets2:
        df = pd.read_excel(xls2, sheet)
        species = sheet.replace("_summary", "")

        if "substrates" not in df.columns:
            raise ValueError(f"Sheet {sheet} in iter2 missing 'substrates'")
        for sub in df["substrates"].dropna().unique():
            consumer_map.setdefault(sub, set()).add(species)

    # ---- remove species that both produce & consume m ----
    # do this before selecting metabolites
    for m in set(producer_map.keys()) & set(consumer_map.keys()):
        overlap = producer_map[m] & consumer_map[m]
        if overlap:
            producer_map[m] = producer_map[m] - overlap
            consumer_map[m] = consumer_map[m] - overlap

    # ---- metabolites bridging iter1 -> iter2 and new in iter1 ----
    metabolites = []
    for m in producer_map:
        if m not in consumer_map:
            continue
        if m in substrates_iter1:
            continue  # was already in medium in iter1, not "new"
        if len(producer_map[m]) < min_producers:
            continue
        if len(consumer_map[m]) < min_consumers:
            continue
        metabolites.append(m)

    # ---- Classification + names ----
    xls_comp = pd.ExcelFile(compound_class_path)
    comp_graph = pd.read_excel(xls_comp, "Graphing")

    class_map = dict(zip(comp_graph["KEGG ID"], comp_graph["Classification"]))
    name_map  = dict(zip(comp_graph["KEGG ID"], comp_graph["Name"]))

    rows = []
    for m in metabolites:
        for sp_p in producer_map[m]:
            for sp_c in consumer_map[m]:
                # at this point sp_p and sp_c are never equal for this m
                rows.append({
                    "producer":        sp_p,
                    "metabolite":      m,
                    "metabolite_name": name_map.get(m, m),
                    "class":           class_map.get(m, "Other/Unknown"),
                    "consumer":        sp_c,
                    "value":           1,
                })

    return pd.DataFrame(rows)


edges_df = build_edges(
    SUMMARY1_PATH,
    SUMMARY2_PATH,
    COMPOUND_CLASS_PATH,
    min_producers=MIN_PRODUCERS,
    min_consumers=MIN_CONSUMERS,
)

print("Total edges (all classes):", len(edges_df))
print("Classes:", sorted(edges_df["class"].unique()))

# ============================================================
# B. Species order (all 21, ordered by phylum)
# ============================================================

def build_species_order(edges_df, phyla_dict, phyla_order):
    species_all = sorted(
        set(edges_df["producer"]) |
        set(edges_df["consumer"]) |
        set(phyla_dict.keys())
    )

    mapped   = [sp for sp in species_all if sp in phyla_dict]
    unmapped = [sp for sp in species_all if sp not in phyla_dict]

    mapped_sorted = sorted(
        mapped,
        key=lambda sp: (phyla_order.index(phyla_dict[sp]), sp)
    )
    unmapped_sorted = sorted(unmapped)

    if unmapped_sorted:
        print("Species missing from phyla_dict:", unmapped_sorted)

    return mapped_sorted + unmapped_sorted

species_order = build_species_order(edges_df, phyla_dict, phyla_order)

# ============================================================
# C. Metabolite selection per class (>10)
# ============================================================

def select_metabolites_for_class(edges_class,
                                 max_metabolites=10):
    stats = (
        edges_class
        .groupby("metabolite")
        .agg(
            n_prod=("producer", "nunique"),
            n_cons=("consumer", "nunique"),
        )
        .reset_index()
    )

    if len(stats) <= max_metabolites:
        return set(stats["metabolite"])

    def score_row(row):
        n_prod = row["n_prod"]
        n_cons = row["n_cons"]
        score = 0.0
        if n_prod == 1:
            score += 5
        elif n_prod == 2:
            score += 3
        elif n_prod <= 4:
            score += 1

        if 2 <= n_cons <= 10:
            score += 4
        elif n_cons > 10:
            score += 2
        elif n_cons == 1:
            score += 0.5

        score += 0.1 * n_cons
        return score

    stats["score"] = stats.apply(score_row, axis=1)
    stats = stats.sort_values("score", ascending=False)

    return set(stats["metabolite"].head(max_metabolites))

# ============================================================
# D. Matplotlib Sankey plotter
# ============================================================

def plot_sankey_matplotlib(edges_cls,
                           species_order,
                           phyla_dict,
                           title=""):

    producers  = list(species_order)
    consumers  = list(species_order)
    metabolites = sorted(edges_cls["metabolite"].unique())

    met_name_map = (
        edges_cls
        .groupby("metabolite")["metabolite_name"]
        .first()
        .to_dict()
    )

    nP = len(producers)
    nM = len(metabolites)
    nC = len(consumers)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    node_info = {}

    # left species
    xP = 0.07
    wP = 0.04
    hP = 0.8 / max(1, nP)
    for i, sp in enumerate(producers):
        y = 0.1 + (i + 0.5) * hP
        h = hP * 0.7
        ph = phyla_dict.get(sp, "Unknown")
        color = phyla_color_map.get(ph, phyla_color_map["Unknown"])
        rect = Rectangle((xP, y - h/2), wP, h,
                         linewidth=0, edgecolor="none",
                         facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(xP - 0.01, y, sp, ha="right", va="center", fontsize=8)
        node_info[("P", sp)] = {"x": xP, "y": y, "w": wP, "h": h}

    # metabolites
    xM = 0.5
    wM = 0.09
    hM = 0.8 / max(1, nM)
    for i, m in enumerate(metabolites):
        y = 0.1 + (i + 0.5) * hM
        h = hM * 0.7
        rect = Rectangle((xM - wM/2, y - h/2), wM, h,
                         linewidth=0, edgecolor="none",
                         facecolor="#E6E6E6", zorder=2)
        ax.add_patch(rect)
        label = met_name_map.get(m, m)
        ax.text(xM, y, label, ha="center", va="center", fontsize=9)
        node_info[("M", m)] = {"x": xM - wM/2, "y": y, "w": wM, "h": h}

    # right species
    wC = 0.04
    xC = 0.93 - wC
    hC = 0.8 / max(1, nC)
    for i, sp in enumerate(consumers):
        y = 0.1 + (i + 0.5) * hC
        h = hC * 0.7
        ph = phyla_dict.get(sp, "Unknown")
        color = phyla_color_map.get(ph, phyla_color_map["Unknown"])
        rect = Rectangle((xC, y - h/2), wC, h,
                         linewidth=0, edgecolor="none",
                         facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(xC + wC + 0.01, y, sp, ha="left", va="center", fontsize=8)
        node_info[("C", sp)] = {"x": xC, "y": y, "w": wC, "h": h}

    # flows
    grouped_pm = (
        edges_cls.groupby(["producer", "metabolite"])["value"].sum().reset_index()
    )
    grouped_mc = (
        edges_cls.groupby(["metabolite", "consumer"])["value"].sum().reset_index()
    )

    if len(grouped_pm) + len(grouped_mc) == 0:
        ax.set_title(title, fontsize=14)
        plt.tight_layout()
        return fig, ax

    max_val = max(
        grouped_pm["value"].max() if not grouped_pm.empty else 0,
        grouped_mc["value"].max() if not grouped_mc.empty else 0,
    )
    base_width = EDGE_BASE_WIDTH
    scale = EDGE_SCALE / max(1, max_val)

    def draw_curve(x0, y0, x1, y1, v):
        w = base_width + scale * v
        xm0 = x0 + (x1 - x0) * 0.33
        xm1 = x0 + (x1 - x0) * 0.66
        verts = [(x0, y0), (xm0, y0), (xm1, y1), (x1, y1)]
        codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
        path = Path(verts, codes)
        patch = PathPatch(path, facecolor="none",
                          edgecolor=EDGE_COLOR,
                          linewidth=w, zorder=1)
        ax.add_patch(patch)

    for _, row in grouped_pm.iterrows():
        sp = row["producer"]
        m  = row["metabolite"]
        v  = row["value"]
        if ("P", sp) not in node_info or ("M", m) not in node_info:
            continue
        ni_p = node_info[("P", sp)]
        ni_m = node_info[("M", m)]
        draw_curve(ni_p["x"] + ni_p["w"], ni_p["y"],
                   ni_m["x"], ni_m["y"], v)

    for _, row in grouped_mc.iterrows():
        m  = row["metabolite"]
        sp = row["consumer"]
        v  = row["value"]
        if ("M", m) not in node_info or ("C", sp) not in node_info:
            continue
        ni_m = node_info[("M", m)]
        ni_c = node_info[("C", sp)]
        draw_curve(ni_m["x"] + ni_m["w"], ni_m["y"],
                   ni_c["x"], ni_c["y"], v)

    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    return fig, ax

# ============================================================
# E. Generate & save
# ============================================================

def safe_filename(s: str) -> str:
    s = s.strip().replace(" ", "_")
    return re.sub(r"[^A-Za-z0-9_\-]", "", s) or "figure"

classes_to_plot = sorted(edges_df["class"].unique())

for cls in classes_to_plot:
    edges_cls = edges_df[edges_df["class"] == cls].copy()
    if edges_cls.empty:
        continue

    n_met_full = edges_cls["metabolite"].nunique()
    print(f"\nClass: {cls}  |  metabolites before filter: {n_met_full}")

    if n_met_full > MAX_METABOLITES_PER_CLASS:
        keep = select_metabolites_for_class(edges_cls,
                                            max_metabolites=MAX_METABOLITES_PER_CLASS)
        edges_cls = edges_cls[edges_cls["metabolite"].isin(keep)]
        print("  truncated to:", len(keep))

    if edges_cls.empty:
        print("  (no edges after filtering)")
        continue

    title = f"DiMEs continuous Sankey – {cls} (iter1 new → iter2 uptake, no self)"
    fig, ax = plot_sankey_matplotlib(edges_cls, species_order, phyla_dict, title)

    # show and save
    plt.show()

    fname = f"sankey_iter1new_to_iter2_noSelf_{safe_filename(cls)}.pdf"
    out_path = os.path.join(OUTPUT_DIR, fname)
    fig.savefig(out_path, format="pdf", bbox_inches="tight")
    print("  saved to:", out_path)

    plt.close(fig)
